# MLOps Lab 8: Packaging the Service into a Docker Container

## 1. Aim

To package the delivery-time prediction service from Lab 7 into a Docker container and call the service running inside it.

## 2. Theory

A service that works on one computer can fail on another because of a different Python version, different library versions or a different operating system. **Docker** solves this by packaging the code, the libraries and a small operating system together, so the service runs the same way on every computer.

- **Dockerfile**: a text file with step-by-step instructions for building an image
- **Image**: the package built from the Dockerfile
- **Container**: a running copy of an image

## 3. Procedure Overview

1. Check that Docker is running
2. Write the files for the service
3. Write the Dockerfile
4. Build the image
5. Run the container and call the service
6. Stop the container
7. Complete the task: Containerising the Iris Service

**Before you start.** Open **Docker Desktop** and wait until it shows **Engine running**. The first build downloads files from the internet.

## Step 1: Check That Docker Is Running

Commands starting with `!` are run in the terminal. The output should show both a **Client** and a **Server** section.

In [1]:
import os

# macOS only: lets Jupyter find the docker command (harmless on Windows)
os.environ["PATH"] += os.pathsep + "/Applications/Docker.app/Contents/Resources/bin"

!docker version

Client:
 Version:           29.9.2
 API version:       1.56
 Go version:        go1.26.10
 Git commit:        a82e528
 Built:             Tue Oct 20 18:43:59 2026
 OS/Arch:           darwin/arm64
 Context:           desktop-linux

Server: Docker Desktop 4.96.3 (244472)
 Engine:
  Version:          29.9.2
  API version:      1.56 (minimum version 1.40)
  Go version:       go1.26.10
  Git commit:       0367cbe
  Built:            Tue Oct 20 18:44:14 2026
  OS/Arch:          linux/arm64
  Experimental:     false
 containerd:
  Version:          v2.4.2
  GitCommit:        b7a25261e029d68978bbf290df0a62fc819b3c51
 runc:
  Version:          1.5.2
  GitCommit:        v1.5.2-0-g29dd3dc2
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0


## Step 2: Write the Files for the Service

All files are saved in a folder named `work`. `%%writefile` saves the contents of a cell to a file.

- **`train.py`** trains the delivery-time model and saves it. Docker runs this file while building the image, so the model is created inside the container.
- **`app.py`** is the FastAPI service from Lab 7.
- **`requirements.txt`** lists the libraries the service needs.

In [2]:
os.makedirs("work", exist_ok=True)

In [3]:
%%writefile work/train.py
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression

# Create the delivery dataset (same as Lab 7)
np.random.seed(42)
n = 500
df = pd.DataFrame()
df["distance_km"]   = np.random.uniform(1, 15, n)
df["items"]         = np.random.randint(1, 10, n)
df["traffic_level"] = np.random.randint(1, 4, n)
df["delivery_minutes"] = (10 + 3 * df["distance_km"] + 1.5 * df["items"]
                          + 5 * df["traffic_level"] + np.random.normal(0, 3, n))

# Train and save the model
model = LinearRegression()
model.fit(df[["distance_km", "items", "traffic_level"]], df["delivery_minutes"])
joblib.dump(model, "model.joblib")
print("Model trained and saved")

Writing work/train.py


In [4]:
%%writefile work/app.py
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

model = joblib.load("model.joblib")

app = FastAPI(title="Delivery Time Prediction Service")


class Order(BaseModel):
    distance_km:   float = Field(gt=0, le=15)
    items:         int   = Field(ge=1, le=9)
    traffic_level: int   = Field(ge=1, le=3)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(order: Order):
    row = pd.DataFrame([order.model_dump()])
    prediction = model.predict(row)[0]
    return {"predicted_minutes": round(float(prediction), 1)}

Overwriting work/app.py


In [5]:
%%writefile work/requirements.txt
fastapi
uvicorn
scikit-learn
pandas
joblib

Overwriting work/requirements.txt


## Step 3: Write the Dockerfile

Each line of the Dockerfile is one instruction:

- **`FROM`**: start from a small Linux system with Python 3.11 installed
- **`WORKDIR`**: the folder inside the container where the files go
- **`COPY`**: copy files from the `work` folder into the container
- **`RUN`**: run a command while building the image (install libraries, train the model)
- **`EXPOSE`**: the port the service uses
- **`CMD`**: the command that starts the service when the container runs. `--host 0.0.0.0` allows the service to be reached from outside the container.

In [6]:
%%writefile work/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY train.py app.py ./
RUN python train.py

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting work/Dockerfile


## Step 4: Build the Image

`docker build` follows the Dockerfile and creates an image named `delivery-api` with version `1.0`. **The first build takes a few minutes.** Running it again is much faster, because Docker reuses the steps it has already done.

In [7]:
!docker build -t delivery-api:1.0 work


[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 276B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 276B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 276B                                       0.0s
 => [internal] load metadata for docker

In [8]:
!docker images delivery-api:1.0

                                                            i Info →   U  In Use
IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA
delivery-api:1.0   3c9e0c7d1ee2        755MB          160MB        


## Step 5: Run the Container and Call the Service

`docker run` starts the container in the background (`-d`). `-p 8010:8000` connects port **8010** on your computer to port **8000** inside the container.

In [9]:
!docker run -d --rm --name delivery-lab -p 8010:8000 delivery-api:1.0

c95f5e51a52e67dbf38d1f5d59d2d9d65e5f4e63c2e8301fd5ce564e14ae58ef


We now send requests to the service inside the container. If you see a connection error, wait a few seconds and run the cell again.

In [10]:
import time
import requests

time.sleep(15)   # give the service time to start

print(requests.get("http://127.0.0.1:8010/health").json())

order = {"distance_km": 8, "items": 3, "traffic_level": 2}
print(requests.post("http://127.0.0.1:8010/predict", json=order).json())

{'status': 'ok'}
{'predicted_minutes': 48.4}


You can also open **http://127.0.0.1:8010/docs** in a web browser and try the service there.

## Step 6: Stop the Container

`docker logs` shows the requests the service received. `docker stop` stops the container. Always stop containers you no longer need.

In [11]:
!docker logs delivery-lab
!docker stop delivery-lab

INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     192.168.65.1:46398 - "GET /health HTTP/1.1" 200 OK
INFO:     192.168.65.1:59789 - "POST /predict HTTP/1.1" 200 OK
delivery-lab


# Task: Containerising the Iris Service

Package the **Iris flower classification service** from the Lab 7 task into a Docker container by following the same steps as above. In a new folder named `iris_work`, write a `train.py` that trains a `LogisticRegression(max_iter=1000)` model on the Iris dataset (`from sklearn.datasets import load_iris`) and saves it, an `app.py` with `/health` and `/predict` endpoints that returns the predicted species name, a `requirements.txt`, and a `Dockerfile`. Build the image with the name `iris-api:1.0`, run it as a container named `iris-lab` on port **8020**, call `/predict` once with the `requests` library, and then stop the container. Submit the notebook and a screenshot of the `/docs` page (http://127.0.0.1:8020/docs) showing a successful prediction.